# 手語翻譯 Transformer｜正式訓練 Notebook

這份 Notebook 延續你已通過 **10/10 Greedy Decoding** 的小資料 Overfitting Test，現在改成使用完整 **Training / Validation** 資料進行正式訓練。

**流程：** 讀取 128 維 Keypoints 與文字 Token → 15 FPS 降採樣 → Transformer Encoder-Decoder → 訓練/驗證 → Loss 曲線 → Early Stopping → 儲存最佳模型 → 查看 Validation 翻譯結果。

**重要：** 小資料的 10/10 是「訓練樣本記憶能力」，不是未見影片的翻譯準確率。此 Notebook 會**重新初始化模型**，不沿用 Overfitting Test 的權重。`test.json` 預設不參與訓練或調參，最後才可手動開啟測試。

## 0. 執行前準備

請把 Notebook 放在**專案根目錄**（或其子資料夾），並在 VS Code 選擇安裝好 `torch`、`numpy`、`matplotlib` 的 Python Kernel。

資料夾預期結構：
```text
專案根目錄/
├── rewrite_Phase3_train/
│   └── dataset/
│       ├── train.json
│       ├── val.json
│       ├── test.json
│       └── vocab.json
└── yolo_pose_mediapie/
    └── hand_keypoints/
        ├── part_1_1.npy
        └── ...
```

`train.json` / `val.json` 的每一筆應包含 `video_id`、`npy_path`、`token_ids`、`text`，而 `.npy` 的形狀應是 `(num_frames, 128)`。

**FPS：** 預設 `.npy` 是由 30 FPS 影片取得，`FRAME_STEP=2` 會變成約 15 FPS；如果 `.npy` 已由 15 FPS 影片產生，改成 `1`。

**訓練時間／顯示記憶體：** 因資料是完整影片，部分序列可能很長。第一輪建議 `BATCH_SIZE=1`。如果 CUDA Out of Memory，可縮小模型，或另行評估較低 FPS。

In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# -------------- 資料 --------------
SEED = 42
FRAME_STEP = 2      # 原始 NPY 為 30 FPS -> 約 15 FPS；若已是 15 FPS，改成 1
BATCH_SIZE = 1
NUM_WORKERS = 0     # Windows / Jupyter 建議先使用 0
PAD_ID = 0

# -------------- 模型 --------------
D_MODEL = 192
NHEAD = 4
ENCODER_LAYERS = 3
DECODER_LAYERS = 3
DIM_FF = 512
DROPOUT = 0.2
MAX_LEN = 4096

# -------------- 訓練 --------------
EPOCHS = 100
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 10
MIN_DELTA = 1e-4
LR_PATIENCE = 3
LR_FACTOR = 0.5
USE_AMP = True      # CUDA 時使用混合精度；CPU 時自動關閉
PLOT_EVERY = 1     # 每 N 個 Epoch 更新一次曲線；設為 0 則訓練完成才畫

# -------------- 推論 / 測試 --------------
VALID_PREVIEW_COUNT = 2
MAX_NEW_TOKENS = 160  # 中文 Token 最多生成數量，太小可能使長句截斷
RUN_FINAL_TEST = False # 測試集留到模型設定完全確定後才改 True

# 自動向上搜尋專案根目錄
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'rewrite_Phase3_train' / 'dataset' / 'train.json').is_file()),
    Path.cwd()
)
DATASET_DIR = PROJECT_ROOT / 'rewrite_Phase3_train' / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'rewrite_Phase3_train' / 'training_runs' / 'transformer_full'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp_enabled = USE_AMP and device.type == 'cuda'
print('專案根目錄:', PROJECT_ROOT)
print('輸出資料夾:', OUTPUT_DIR)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('混合精度 AMP:', amp_enabled)

## 1. 讀取 Train / Validation，檢查 Vocabulary 與影片長度

`test.json` **不會在這一節被載入或拿來決定參數**。Vocabulary 沿用你已用 Training Split 建好的 `vocab.json`，不在 Notebook 內重建。先統計長度，避免長影片超過位置編碼長度。

In [ ]:
with open(DATASET_DIR / 'train.json', encoding='utf-8') as f:
    train_rows = json.load(f)
with open(DATASET_DIR / 'val.json', encoding='utf-8') as f:
    val_rows = json.load(f)
with open(DATASET_DIR / 'vocab.json', encoding='utf-8') as f:
    vocab = json.load(f)

vocab_size = len(vocab)
assert vocab.get('<pad>') == PAD_ID
assert len(set(vocab.values())) == vocab_size
assert set(vocab.values()) == set(range(vocab_size)), 'Vocabulary ID 需要連續從 0 開始'

# 原始來源群組用 part_數字 標示，例如 part_1_1 與 part_1_2 同組
def group_id(row):
    return '_'.join(row['video_id'].split('_')[:2])

train_groups = {group_id(r) for r in train_rows}
val_groups = {group_id(r) for r in val_rows}
assert not (train_groups & val_groups), 'Training / Validation 存在相同來源群組'


def collect_stats(rows):
    frames, token_counts = [], []
    for row in rows:
        filename = PROJECT_ROOT / row['npy_path']
        n_frames, n_features = np.load(filename, mmap_mode='r').shape
        assert n_features == 128, f'{filename} 特徵維度不是 128'
        frames.append((n_frames + FRAME_STEP - 1) // FRAME_STEP)
        token_counts.append(len(row['token_ids']))
    return np.array(frames), np.array(token_counts)

train_frames, train_tokens = collect_stats(train_rows)
val_frames, val_tokens = collect_stats(val_rows)

for name, frames, tokens in [
    ('Training', train_frames, train_tokens),
    ('Validation', val_frames, val_tokens)
]:
    print(f'{name}: {len(frames)} 筆 | Frame 長度 最小/中位/最大 = '
          f'{frames.min()}/{np.median(frames):.0f}/{frames.max()} | '
          f'Token 長度 最小/中位/最大 = '
          f'{tokens.min()}/{np.median(tokens):.0f}/{tokens.max()}')

print('Vocabulary Size:', vocab_size)
assert max(train_frames.max(), val_frames.max(), train_tokens.max(), val_tokens.max()) <= MAX_LEN, (
    f'資料序列長度超過 MAX_LEN={MAX_LEN}；請增加 MAX_LEN 或調整降採樣，並評估顯存。'
)
assert min(train_frames.min(), val_frames.min()) > 0
assert min(train_tokens.min(), val_tokens.min()) >= 2

## 2. 建立 Dataset / DataLoader 與 Attention Masks

`src_key_padding_mask` 用來區別不同長度影片的 Padding，**不等於**你手部特徵最後兩欄的左右手 Presence Mask。前者是 `(B, T)`；後者保留在 128 維輸入中。

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, rows, project_root=PROJECT_ROOT, frame_step=FRAME_STEP):
        self.rows = rows
        self.project_root = project_root
        self.frame_step = frame_step

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        src = np.load(self.project_root / row['npy_path'])[::self.frame_step].astype(
            np.float32, copy=True
        )
        tgt = np.asarray(row['token_ids'], dtype=np.int64)
        return torch.from_numpy(src), torch.from_numpy(tgt)


def collate_fn(batch):
    sources, targets = zip(*batch)
    src_lens = torch.tensor([len(s) for s in sources])
    src = pad_sequence(sources, batch_first=True, padding_value=0.0)
    tgt = pad_sequence(targets, batch_first=True, padding_value=PAD_ID)

    src_kpm = torch.arange(src.size(1))[None, :] >= src_lens[:, None]
    tgt_in = tgt[:, :-1]
    tgt_out = tgt[:, 1:]
    tgt_kpm = tgt_in == PAD_ID
    target_len = tgt_in.size(1)
    causal_mask = torch.triu(
        torch.ones((target_len, target_len), dtype=torch.bool), diagonal=1
    )
    return src, src_kpm, tgt_in, tgt_out, tgt_kpm, causal_mask

train_dataset = SignLanguageDataset(train_rows)
val_dataset = SignLanguageDataset(val_rows)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=NUM_WORKERS
)

sample_batch = next(iter(val_loader))
for name, tensor in zip(
    ('src', 'src_kpm', 'tgt_in', 'tgt_out', 'tgt_kpm', 'causal_mask'),
    sample_batch
):
    print(name, tuple(tensor.shape))

## 3. Transformer Encoder-Decoder

沿用之前成功的架構，`input_proj: 128 → D_MODEL`；Decoder 預測 Vocabulary 中的下一個 Token。

這裡設定 `enable_nested_tensor=False`，避免不同 PyTorch 版本中 Nested Tensor 的實驗性警告；不會改變輸入輸出格式。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SignTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=192, nhead=4,
                 num_encoder_layers=3, num_decoder_layers=3, dim_ff=512,
                 dropout=0.2, pad_id=0, max_len=4096):
        super().__init__()
        self.input_proj = nn.Linear(128, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                dropout=dropout, batch_first=True
            ), num_layers=num_encoder_layers, enable_nested_tensor=False
        )
        self.decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
                dropout=dropout, batch_first=True
            ), num_layers=num_decoder_layers
        )
        self.output_proj = nn.Linear(d_model, vocab_size)

    def encode(self, src, src_kpm):
        x = self.pos_enc(self.input_proj(src))
        return self.encoder(x, src_key_padding_mask=src_kpm)

    def decode_logits(self, memory, src_kpm, tgt_in, tgt_kpm, causal_mask):
        tgt_emb = self.pos_enc(self.tgt_embed(tgt_in))
        decoded = self.decoder(
            tgt_emb, memory, tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_kpm, memory_key_padding_mask=src_kpm
        )
        return self.output_proj(decoded)

    def forward(self, src, tgt_in, src_kpm, tgt_kpm, causal_mask):
        memory = self.encode(src, src_kpm)
        return self.decode_logits(memory, src_kpm, tgt_in, tgt_kpm, causal_mask)


MODEL_CONFIG = {
    'vocab_size': vocab_size, 'd_model': D_MODEL, 'nhead': NHEAD,
    'num_encoder_layers': ENCODER_LAYERS, 'num_decoder_layers': DECODER_LAYERS,
    'dim_ff': DIM_FF, 'dropout': DROPOUT, 'pad_id': PAD_ID,
    'max_len': MAX_LEN
}

# 全新模型：不使用先前單筆或 10 筆測試的權重
model = SignTransformer(**MODEL_CONFIG).to(device)
print('模型參數量:', f'{sum(p.numel() for p in model.parameters()):,}')

batch = tuple(t.to(device) for t in sample_batch)
model.eval()
with torch.inference_mode():
    output = model(batch[0], batch[2], batch[1], batch[4], batch[5])
print('Forward Pass:', tuple(output.shape))
assert tuple(output.shape) == (batch[0].size(0), batch[2].size(1), vocab_size)
del output, batch

## 4. Optimizer、Loss、AMP 與學習率排程

- `CrossEntropyLoss`：忽略 `<pad>`；依有效 Target Token 數做加權平均。
- `AdamW` + `ReduceLROnPlateau`：Validation Loss 停滯時降低學習率。
- `Gradient Clipping`：上限 1.0。
- `AMP`：有 CUDA 時啟用 Float16 混合精度；若你的 GPU/驅動不穩，可設 `USE_AMP=False` 後重新執行整份 Notebook。

**提醒：** `LABEL_SMOOTHING=0.1` 時 Loss 不一定能接近 0，正式訓練重點是 Validation Loss，而非只看 Training Loss。

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE
)
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)


def token_loss_sum(logits, target):
    loss_sum = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), target.reshape(-1),
        ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING, reduction='sum'
    )
    count = (target != PAD_ID).sum()
    return loss_sum, count


def run_epoch(loader, training):
    model.train(training)
    total_loss, total_tokens = 0.0, 0

    grad_context = torch.enable_grad() if training else torch.no_grad()
    with grad_context:
        for batch in loader:
            src, src_kpm, tgt_in, tgt_out, tgt_kpm, causal = (
                tensor.to(device) for tensor in batch
            )
            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(src, tgt_in, src_kpm, tgt_kpm, causal)
                loss_sum, valid_tokens = token_loss_sum(logits, tgt_out)
                loss = loss_sum / valid_tokens

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss_sum.detach().item()
            total_tokens += valid_tokens.item()

    return total_loss / total_tokens

print('Optimizer 與 Training Loop 已建立')

## 5. 開始正式訓練（Train + Validation）

每個 Epoch：訓練全部 Training → 評估全部 Validation → 更新 Learning Rate → 如有改善就儲存 `best_model.pth`。

- `best_model.pth`：Validation Loss 最佳的模型，可用來做推論。
- `last_checkpoint.pth`：最後完成 Epoch 的狀態，便於保留紀錄。
- `history.json`、`loss_curve.png`：每次 Epoch 更新。
- Validation Loss **連續 10 個 Epoch 未改善**，就 Early Stop。

**注意：** 目前 Notebook 每次重新執行訓練 Cell，都會從當下記憶體中的模型接續訓練；若要乾淨的全新實驗，請 Restart Kernel → Run All。

In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / 'best_model.pth'
LAST_MODEL_PATH = OUTPUT_DIR / 'last_checkpoint.pth'
HISTORY_PATH = OUTPUT_DIR / 'history.json'
CURVE_PATH = OUTPUT_DIR / 'loss_curve.png'

history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'lr': []}
best_val_loss = float('inf')
best_epoch = 0
no_improve = 0
train_start = time.perf_counter()


def draw_losses(hist, save_path=None, show=True):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(hist['epoch'], hist['train_loss'], marker='.', label='Training Loss')
    ax.plot(hist['epoch'], hist['val_loss'], marker='.', label='Validation Loss')
    ax.set(xlabel='Epoch', ylabel='Token-weighted Cross-Entropy Loss',
           title='Transformer Training / Validation Loss')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150)
    if show:
        display(fig)
    plt.close(fig)


for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]['lr']

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(lr)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch
        no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'model_config': MODEL_CONFIG,
            'vocab': vocab,
            'frame_step': FRAME_STEP,
            'val_loss': val_loss,
            'label_smoothing': LABEL_SMOOTHING
        }, BEST_MODEL_PATH)
    else:
        no_improve += 1

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'model_config': MODEL_CONFIG,
        'vocab': vocab,
        'frame_step': FRAME_STEP,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'history': history
    }, LAST_MODEL_PATH)

    with open(HISTORY_PATH, 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

    draw_losses(history, save_path=CURVE_PATH, show=False)
    line = (
        f'Epoch {epoch:3d}/{EPOCHS} | Train={train_loss:.4f} | Val={val_loss:.4f} | '
        f'Best={best_val_loss:.4f} (Epoch {best_epoch}) | LR={lr:.2e} | '
        f'{time.perf_counter() - epoch_start:.1f}s'
    )
    if PLOT_EVERY and (epoch == 1 or epoch % PLOT_EVERY == 0):
        clear_output(wait=True)
        print(line)
        print(f'無改善次數：{no_improve}/{EARLY_STOP_PATIENCE}')
        draw_losses(history, show=True)
    else:
        print(line)

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f'Early Stopping：Validation Loss 已連續 {EARLY_STOP_PATIENCE} 個 Epoch 未改善。')
        break

print('\n正式訓練已結束')
print('實際 Epoch:', history['epoch'][-1])
print('最佳 Epoch:', best_epoch)
print('最佳 Validation Loss:', f'{best_val_loss:.4f}')
print('總耗時（分鐘）:', f'{(time.perf_counter() - train_start) / 60:.1f}')
print('最佳模型:', BEST_MODEL_PATH)
print('Loss 圖表:', CURVE_PATH)

## 6. 載入最佳模型，進行 Validation Greedy Decoding

**務必載入最佳 Validation Checkpoint，不要直接使用最後 Epoch 的模型。**

這裡 Decoder 只拿 `<bos>` 起始，不使用完整正確句子來生成；Encoder 僅計算一次，以避免每生成一個 Token 就重複運算整段影片。

`MAX_NEW_TOKENS` 是固定推論上限，不依單筆 Validation 正確文字決定。如果長句被截斷，可以加大此值，但推論會更慢。

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
best_model = SignTransformer(**checkpoint['model_config']).to(device)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model.eval()

BOS_ID = vocab['<bos>']
EOS_ID = vocab['<eos>']
ID_TO_TOKEN = {i: token for token, i in vocab.items()}


def to_text(ids):
    result = []
    for idx in ids:
        idx = int(idx)
        if idx == EOS_ID:
            break
        if idx not in (BOS_ID, PAD_ID):
            result.append(ID_TO_TOKEN.get(idx, '<unk>'))
    return ''.join(result)


def output_ids(ids):
    ids = [int(i) for i in ids]
    if ids and ids[0] == BOS_ID:
        ids = ids[1:]
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID) + 1]
    return [i for i in ids if i != PAD_ID]


@torch.inference_mode()
def greedy_decode_one(model, src, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    src = src.unsqueeze(0).to(device)
    src_kpm = torch.zeros((1, src.size(1)), dtype=torch.bool, device=device)
    memory = model.encode(src, src_kpm)
    generated = torch.full((1, 1), BOS_ID, device=device, dtype=torch.long)

    for _ in range(max_new_tokens):
        t = generated.size(1)
        causal = torch.triu(
            torch.ones((t, t), dtype=torch.bool, device=device), diagonal=1
        )
        logits = model.decode_logits(
            memory, src_kpm, generated, generated.eq(PAD_ID), causal
        )
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        if next_id.item() == EOS_ID:
            break
    return generated[0].tolist()


print('最佳模型：Epoch', checkpoint['epoch'], 'Validation Loss:', checkpoint['val_loss'])
print('Validation 範例（僅供人工觀察，不能代表整體泛化能力）：')
for idx in range(min(VALID_PREVIEW_COUNT, len(val_dataset))):
    src, target = val_dataset[idx]
    predicted = greedy_decode_one(best_model, src)
    reference = output_ids(target.tolist())
    print('\n影片:', val_rows[idx]['video_id'])
    print('原始字幕:', val_rows[idx]['text'])
    print('可還原標籤:', to_text(reference))
    print('預測文字:', to_text(predicted))
    print('Token IDs 完全一致:', output_ids(predicted) == reference)

## 7. 可選：全部 Validation Greedy Decoding 統計

這一步是**完整自回歸生成**，不是 Teacher Forcing 計算的 Loss，對長影片而言可能比較耗時。預設關閉，需要時改成 `RUN_FULL_VALIDATION_DECODE=True` 再執行此 Cell。

用 Token Exact Match 作為初步統計；對手語翻譯而言，後續仍建議補上 chrF、BLEU 與人工檢查，因為正確語意可能有不同說法。

In [ ]:
RUN_FULL_VALIDATION_DECODE = False

if RUN_FULL_VALIDATION_DECODE:
    full_results = []
    for idx, row in enumerate(val_rows):
        src, target = val_dataset[idx]
        pred = greedy_decode_one(best_model, src)
        match = output_ids(pred) == output_ids(target.tolist())
        full_results.append({
            'video_id': row['video_id'],
            'reference': to_text(target.tolist()),
            'prediction': to_text(pred),
            'exact_match': match
        })
        print(f'{idx+1}/{len(val_rows)} {row["video_id"]}: Exact Match={match}')
    with open(OUTPUT_DIR / 'validation_predictions.json', 'w', encoding='utf-8') as f:
        json.dump(full_results, f, ensure_ascii=False, indent=2)
    print('Validation 完全一致:', sum(x['exact_match'] for x in full_results), '/', len(full_results))
else:
    print('已略過完整 Validation Greedy Decoding。')

## 8. 最終測試（預設關閉）

只有當你**不再依據 Test Set 結果調整模型設定**時，才將最上方的 `RUN_FINAL_TEST` 改成 `True`。此 Cell 使用最佳 Validation Checkpoint 在 `test.json` 上產生推論結果，並輸出 `test_predictions.json`。

注意：`<unk>` 代表 Training Vocabulary 無法表示的詞彙，因此生成文字與原始字幕可能不同。

In [ ]:
if RUN_FINAL_TEST:
    with open(DATASET_DIR / 'test.json', encoding='utf-8') as f:
        test_rows = json.load(f)
    test_groups = {group_id(row) for row in test_rows}
    assert not (test_groups & train_groups)
    assert not (test_groups & val_groups)

    test_dataset = SignLanguageDataset(test_rows)
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=NUM_WORKERS
    )
    # 使用最佳模型評估 Test teacher-forced Loss
    previous_model = model
    model = best_model
    test_loss = run_epoch(test_loader, training=False)
    model = previous_model
    print('Test Loss:', f'{test_loss:.4f}')

    test_results = []
    for idx, row in enumerate(test_rows):
        src, target = test_dataset[idx]
        pred = greedy_decode_one(best_model, src)
        test_results.append({
            'video_id': row['video_id'],
            'original_text': row['text'],
            'vocab_reference': to_text(target.tolist()),
            'prediction': to_text(pred),
            'exact_match': output_ids(pred) == output_ids(target.tolist())
        })
        print(f'{idx+1}/{len(test_rows)} {row["video_id"]} 已完成')

    with open(OUTPUT_DIR / 'test_predictions.json', 'w', encoding='utf-8') as f:
        json.dump(test_results, f, ensure_ascii=False, indent=2)
    print('Test Token 完全一致:', sum(r['exact_match'] for r in test_results), '/', len(test_results))
else:
    print('最終測試集尚未使用（RUN_FINAL_TEST=False），保留作最後評估。')

## 9. 如何解讀結果？

- **Training Loss 降、Validation Loss 也降**：模型可能開始學到有助於泛化的規律。
- **Training Loss 降、Validation Loss 升**：可能過度擬合，應以 Validation Loss 最佳的 Checkpoint 為準。
- **所有未見影片都輸出相似文字**：可能發生常見句子偏差或 Decoder 依賴語言模式，需要檢查預測與 Keypoints 對照。
- **Validation Loss 下降不等於翻譯準確**：仍要檢查 Greedy Decoding、中文字品質、句子含義；最好用 chrF / BLEU 等指標搭配人工評估。
- **字幕與手語內容大致配對即可做實驗，但若存在遺漏/時間不對齊仍會影響泛化。** 你目前的 128 維手部正規化也移除了手腕在畫面中的絕對位置，因此不能保證能處理全部手語語意。

### 儲存檔案
```text
rewrite_Phase3_train/training_runs/transformer_full/
├── best_model.pth
├── last_checkpoint.pth
├── history.json
├── loss_curve.png
├── validation_predictions.json   # 只有開啟全部 Validation 解碼才會有
└── test_predictions.json         # 只有最終測試開啟後才會有
```

先觀察正式訓練的 Loss 曲線與 Validation 預測，再決定是否縮小模型、改良 Keypoints 或擴充資料。